# Faithful h2oloo Reproduction (Pradeep et al., SIGIR 2022)

Reproduces the winning-architecture pipeline **using their own code and toolkits** so the
comparison is apples-to-apples: NQS (doc2query-T5) → BM25+RM3 (Pyserini) → RRF → monoT5 (PyGaggle).

**Fidelity anchors (from their pyserini reproduction doc, verified from source):**
- Corpus `contents` = `title + condition[first] + summary + detailed_description + eligibility`
  (their `convert_trec21_ct_to_json.py`, downloaded and used verbatim — NOT our full-text corpus,
  which differs: we add interventions/all-conditions/labels).
- BM25 `k1=0.9, b=0.4`; RM3 defaults. Topics via their `convert_topic_xml_to_tsv.py`.
- **Validation target (TREC 2021):** BM25 nDCG@10 = **0.2923**, BM25+RM3 = **0.3539** (their doc).
  If we don't hit these, the reproduction is not faithful — fix before proceeding.
- Metrics: `trec_eval -c -m ndcg_cut` (graded) and `-c -l 2` for P@10/RR (eligible-only).

In [ ]:
# Pyserini (their retrieval toolkit) needs JVM 11; PyGaggle for monoT5.
!apt-get -qq install -y openjdk-11-jdk-headless > /dev/null
import os; os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
!pip install -q pyserini==0.22.1 faiss-cpu ftfy
!pip install -q git+https://github.com/castorini/pygaggle.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/h2oloo'; os.makedirs(WORK, exist_ok=True)
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
os.chdir(WORK)

In [ ]:
# Use THEIR exact conversion scripts (downloaded from castorini/pyserini) — no reimplementation.
BASE = 'https://raw.githubusercontent.com/castorini/pyserini/master/scripts/trec-ct'
!wget -q $BASE/convert_trec21_ct_to_json.py -O convert_ct_to_json.py
!wget -q $BASE/convert_topic_xml_to_tsv.py  -O convert_topic_xml_to_tsv.py
print('their scripts fetched')

In [ ]:
# TREC 2021 corpus + topics (the fidelity-validation year). Uses their pipeline verbatim.
os.makedirs('collections/trec-ct', exist_ok=True)
for i in range(1, 6):
    u = f'http://www.trec-cds.org/2021_data/ClinicalTrials.2021-04-27.part{i}.zip'
    !wget -q -c $u -P collections/trec-ct
!unzip -qo 'collections/trec-ct/*.zip' -d collections/trec-ct
!wget -q http://www.trec-cds.org/topics2021.xml -O topics2021.xml
!wget -q https://trec.nist.gov/data/trials/qrels2021.txt -O qrels2021.txt || \
 wget -q https://trec.nist.gov/data/trials/2021-qrels.txt -O qrels2021.txt
!python convert_ct_to_json.py --input_dir collections/trec-ct --output_dir collections/trec-ct-json
!python convert_topic_xml_to_tsv.py --topics topics2021.xml --queries ctqueries2021.tsv
print('indexed docs expected: 375,580')

In [ ]:
!python -m pyserini.index --collection JsonCollection \
 --generator DefaultLuceneDocumentGenerator --threads 9 --input collections/trec-ct-json \
 --index indexes/lucene-index-ct --storePositions --storeDocvectors --storeRaw

In [ ]:
# BM25 and BM25+RM3 with their exact params; validate against their published numbers.
!python -m pyserini.search --topics ctqueries2021.tsv --index indexes/lucene-index-ct \
 --output runs/bm25.txt --hits 1000 --bm25 --k1 0.9 --b 0.4
!python -m pyserini.search --topics ctqueries2021.tsv --index indexes/lucene-index-ct \
 --output runs/bm25.rm3.txt --hits 1000 --bm25 --rm3 --k1 0.9 --b 0.4
import pytrec_eval, json
!pip install -q pytrec_eval
def evalr(runfile):
    q = {}
    for l in open('qrels2021.txt'):
        t,_,d,r = l.split(); q.setdefault(t,{})[d]=int(r)
    run = {}
    for l in open(runfile):
        t,_,d,rk,s,_ = l.split(); run.setdefault(t,{})[d]=float(s)
    import numpy as np
    ev = pytrec_eval.RelevanceEvaluator(q, {'ndcg_cut.10'}).evaluate(run)
    return np.mean([v['ndcg_cut_10'] for v in ev.values()])
print(f'BM25     nDCG@10 = {evalr("runs/bm25.txt"):.4f}   (their doc: 0.2923)')
print(f'BM25+RM3 nDCG@10 = {evalr("runs/bm25.rm3.txt"):.4f}   (their doc: 0.3539)')
print('\nIf these match, the corpus/index/retrieval reproduction is faithful.')

## Neural Query Synthesis (their NQS)
doc2query-T5 generates ~40 single-sentence queries per patient note; each is issued to BM25+RM3
and the lists are RRF-fused. Their model: doc2query–T5 (they use the T5-3B trained on MS MARCO V2).
Public checkpoint: `castorini/doc2query-t5-base-msmarco` (base) or the 3B if available. Match the
settings from the paper: top-k sampling k=10, ~40 queries, max 512 in / 64 out tokens.

In [ ]:
import torch, re
from transformers import T5Tokenizer, T5ForConditionalGeneration
D2Q = 'castorini/doc2query-t5-base-msmarco'   # paper used T5-3B (MS MARCO V2); base is the public match
d2q_tok = T5Tokenizer.from_pretrained(D2Q)
d2q = T5ForConditionalGeneration.from_pretrained(D2Q).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
N_QUERIES = 40

@torch.no_grad()
def synth_queries(patient_text, n=N_QUERIES):
    ids = d2q_tok.encode(patient_text, return_tensors='pt', truncation=True, max_length=512).to(d2q.device)
    out = d2q.generate(ids, max_length=64, do_sample=True, top_k=10, num_return_sequences=n)
    qs = [d2q_tok.decode(o, skip_special_tokens=True).strip() for o in out]
    return [re.sub('\\s\\s+', ' ', q) for q in qs if q]

# demo on one topic
topics = {l.split('\t')[0]: l.split('\t')[1].strip() for l in open('ctqueries2021.tsv')}
_t = list(topics.values())[0]
print('sample synthesized queries:')
for q in synth_queries(_t, 6): print('  -', q)

In [ ]:
# Issue each synthesized query to BM25+RM3 via Pyserini's LuceneSearcher; RRF-fuse per topic.
from pyserini.search.lucene import LuceneSearcher
from collections import defaultdict
from tqdm.auto import tqdm
searcher = LuceneSearcher('indexes/lucene-index-ct')
searcher.set_bm25(0.9, 0.4); searcher.set_rm3()

def rrf(lists, k=60, depth=1000):
    s = defaultdict(float)
    for lst in lists:
        for rank, docid in enumerate(lst):
            s[docid] += 1.0/(k+rank+1)
    return [d for d,_ in sorted(s.items(), key=lambda x:-x[1])[:depth]]

nqs_run = {}
for tid, text in tqdm(topics.items(), desc='NQS retrieve'):
    queries = [text] + synth_queries(text)          # +PD: include the raw note (their best variant)
    lists = []
    for q in queries:
        hits = searcher.search(q, k=1000)
        lists.append([h.docid for h in hits])
    fused = rrf(lists)
    nqs_run[tid] = {d: 1.0/(r+1) for r, d in enumerate(fused)}
with open('runs/nqs.txt','w') as f:
    for t,dd in nqs_run.items():
        for r,(d,_) in enumerate(sorted(dd.items(),key=lambda x:-x[1])):
            f.write(f'{t} Q0 {d} {r+1} {1.0/(r+1):.6f} nqs\n')
print(f'NQS+PD nDCG@10 = {evalr("runs/nqs.txt"):.4f}   (their TREC21 first-stage ~0.4726)')

## monoT5 reranking (their PyGaggle reranker)
Rerank the NQS candidate pool with monoT5. Paper uses monoT5-3B → Med-MARCO (`monoT5_MED`,
zero-shot) → fine-tuned on KZ (`monoT5_CT`). Public checkpoints: `castorini/monot5-3b-msmarco`,
`castorini/monot5-3b-med-msmarco` (the MED variant). The KZ-tuned `monoT5_CT` isn't public, so the
faithful-but-reproducible target is monoT5-3B-MED (their zero-shot row, TREC21 ~0.4715); note this
in the writeup as the reproducible approximation of their winning KZ-tuned run.
Their domain templates: `Query: {patient}  Document: title:{t} condition:{c} eligibility:{e}` (MaxP).

In [ ]:
from pygaggle.rerank.base import Query, Text
from pygaggle.rerank.transformer import MonoT5
# Faithful reproduction: use monoT5_CT (KZ-fine-tuned by finetune_monot5_ct.ipynb) to match their
# winning run's ~0.71 on TREC21. Fall back to the public MED variant (~0.4715) only if not yet trained.
MONOT5_CKPT = '/content/drive/MyDrive/ct_data23/monot5_ct'   # <- trained checkpoint
import os as _os
if not _os.path.exists(MONOT5_CKPT):
    MONOT5_CKPT = 'castorini/monot5-3b-med-msmarco'
    print('WARNING: using public MED variant (~0.4715) — train monoT5_CT first for a faithful ~0.71 reproduction')
reranker = MonoT5(MONOT5_CKPT)
RERANK_K = 100

def field(docid, name):
    import json as _j
    return _j.loads(searcher.doc(docid).raw()).get(name, '')

mono_run = {}
for tid, text in tqdm(topics.items(), desc='monoT5 rerank'):
    head = [d for d, _ in sorted(nqs_run[tid].items(), key=lambda x: -x[1])[:RERANK_K]]
    # their domain-specific ranking template (title/condition/eligibility)
    passages = [Text(f"title: {field(d,'title')} condition: {field(d,'condition')} "
                     f"eligibility: {field(d,'eligibility')}", {'docid': d}, 0) for d in head]
    ranked = reranker.rerank(Query(text), passages)
    mono_run[tid] = {r.metadata['docid']: float(r.score) for r in ranked}
    for r, (d, _) in enumerate(sorted(nqs_run[tid].items(), key=lambda x: -x[1])):
        if d not in mono_run[tid]:
            mono_run[tid][d] = -1e6 - r
with open('runs/monot5.txt', 'w') as f:
    for t, dd in mono_run.items():
        for r, (d, s) in enumerate(sorted(dd.items(), key=lambda x: -x[1])):
            f.write(f'{t} Q0 {d} {r+1} {s:.6f} monot5\n')
print(f'NQS -> monoT5 nDCG@10 = {evalr("runs/monot5.txt"):.4f}')
print('validate: monoT5_CT should reach ~0.7118 on TREC21 (their winning-run number)')

## Notes for the comparison
- **Faithfulness:** validated at the BM25/RM3 anchors (0.2923 / 0.3539). NQS uses public doc2query
  (base, not their private 3B); monoT5 uses the public MED checkpoint (their KZ-tuned `monoT5_CT` is
  not released). State both as reproducible approximations — the *architecture* is faithful.
- **For the study:** run this same pipeline on TREC22 and TREC23 (swap the corpus/topics/qrels), so
  "theirs" is a run we control → paired significance vs ours and the combined system, no NIST run needed.
- **Combined system:** feed NQS queries into *our* hybrid (BM25+dense) pool and add the monoT5 score
  as a feature alongside our clf/v2/LLM in the LambdaMART ensemble.